# Building an Email-Aware Support Agent with Claude and Commune

This notebook shows how to build a **Claude-powered email agent** that can:

- Read and triage an incoming support inbox
- Identify which threads need urgent replies
- Search for specific conversations
- Draft and send replies — all autonomously

We use the [Commune](https://commune.email) API for programmatic inbox access and email delivery,
and the Anthropic Python SDK for the agent loop with native tool use.

**What you'll learn:**
- How to define tools in Anthropic's JSON schema format
- How to implement a correct `tool_use` / `tool_result` loop
- How to wire real API calls into a Claude agent
- A polling pattern for continuous inbox monitoring

**Prerequisites:** An Anthropic API key and a Commune API key.
Set them as environment variables `ANTHROPIC_API_KEY` and `COMMUNE_API_KEY`.


## 1. Setup


In [ ]:
%pip install -q anthropic commune-mail


In [ ]:
import os
import json
import time
import threading
import anthropic
from commune import Commune

# Initialise clients — keys are read from the environment
claude  = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
commune = Commune(api_key=os.environ["COMMUNE_API_KEY"])

MODEL = "claude-opus-4-5"   # or claude-sonnet-4-5 for lower cost
print("Clients ready.")


## 2. Email Tool Functions

These are plain Python functions that wrap the Commune SDK.
The agent will call them whenever Claude decides it needs email data.


In [ ]:
def read_inbox(limit: int = 10, unread_only: bool = False) -> dict:
    """Fetch recent emails from the agent's Commune inbox."""
    emails = commune.emails.list(limit=limit, unread_only=unread_only)
    return {
        "emails": [
            {
                "id":           e["id"],
                "from":         e["from_address"],
                "subject":      e["subject"],
                "preview":      e["body"][:300],   # first 300 chars
                "received_at":  e["received_at"],
                "read":         e["read"],
            }
            for e in emails
        ],
        "total": len(emails),
    }


def get_email(email_id: str) -> dict:
    """Retrieve the full body of a single email by its ID."""
    email = commune.emails.get(email_id)
    return {
        "id":          email["id"],
        "from":        email["from_address"],
        "subject":     email["subject"],
        "body":        email["body"],
        "received_at": email["received_at"],
        "read":        email["read"],
    }


def search_emails(query: str, limit: int = 10) -> dict:
    """Full-text search across the inbox."""
    results = commune.emails.search(query=query, limit=limit)
    return {
        "results": [
            {
                "id":          e["id"],
                "from":        e["from_address"],
                "subject":     e["subject"],
                "preview":     e["body"][:300],
                "received_at": e["received_at"],
            }
            for e in results
        ],
        "total": len(results),
    }


def send_email(to: str, subject: str, body: str) -> dict:
    """Send an email reply via Commune."""
    result = commune.emails.send(to=to, subject=subject, body=body)
    return {"status": "sent", "message_id": result.get("id")}


# Dispatch table — maps tool name → Python function
TOOL_MAP = {
    "read_inbox":    read_inbox,
    "get_email":     get_email,
    "search_emails": search_emails,
    "send_email":    send_email,
}

print("Tool functions defined:", list(TOOL_MAP))


## 3. Tool Definitions (Anthropic Schema)

We describe each tool using Anthropic's JSON schema format so Claude knows
when to call each one and what arguments to pass.


In [ ]:
tools = [
    {
        "name": "read_inbox",
        "description": (
            "Read recent emails from the agent's support inbox. "
            "Use this when the user asks about incoming mail, wants a summary of "
            "unread messages, or needs to triage new support requests."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of emails to return (1–50).",
                    "default": 10,
                },
                "unread_only": {
                    "type": "boolean",
                    "description": "If true, return only unread messages.",
                    "default": False,
                },
            },
            "required": [],
        },
    },
    {
        "name": "get_email",
        "description": (
            "Retrieve the full body of a single email by its ID. "
            "Call this after read_inbox to get the complete content of a specific message."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "email_id": {
                    "type": "string",
                    "description": "The unique ID of the email to retrieve.",
                },
            },
            "required": ["email_id"],
        },
    },
    {
        "name": "search_emails",
        "description": (
            "Full-text search across all emails in the inbox. "
            "Use this to find messages related to a specific topic, customer, or keyword."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query (keywords, email address, phrase).",
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of results (1–50).",
                    "default": 10,
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "send_email",
        "description": (
            "Send an email via Commune. "
            "Use this to reply to a customer or send a follow-up message. "
            "Always confirm the recipient and content before calling this tool."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "Recipient email address.",
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line.",
                },
                "body": {
                    "type": "string",
                    "description": "Plain-text email body.",
                },
            },
            "required": ["to", "subject", "body"],
        },
    },
]

print(f"{len(tools)} tools registered.")


## 4. The Agent Loop

The core of the agent is a loop that:

1. Sends the conversation to Claude (with tools attached)
2. Checks whether Claude wants to call a tool (`stop_reason == "tool_use"`)
3. Executes every requested tool call and collects the results
4. Appends those results as `tool_result` blocks and loops back
5. Returns Claude's final text when `stop_reason == "end_turn"`

This is the standard Anthropic tool-use agentic loop — no framework required.


In [ ]:
SYSTEM_PROMPT = """\
You are a helpful support-inbox assistant. You have access to tools that let you
read, search, and reply to emails in a Commune inbox.

Guidelines:
- Be concise and direct.
- When asked to triage emails, read the inbox first, then retrieve full details
  for any message that looks urgent or complex.
- Before sending any email, summarise what you plan to send and why.
- Never fabricate email content — always base replies on actual message bodies.
"""


def run_agent(user_message: str, verbose: bool = True) -> str:
    """Run the Claude email agent for a single user request.

    Returns the final assistant reply as a string.
    """
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = claude.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
        )

        if verbose:
            print(f"[agent] stop_reason={response.stop_reason}")

        # ── Final answer ────────────────────────────────────────────────────
        if response.stop_reason == "end_turn":
            # Extract the text block from the response
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text in response)"

        # ── Tool calls ──────────────────────────────────────────────────────
        if response.stop_reason == "tool_use":
            # Add Claude's response (which includes the tool_use blocks) to history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each requested tool and collect results
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_name = block.name
                tool_input = block.input

                if verbose:
                    print(f"  → calling {tool_name}({tool_input})")

                try:
                    result = TOOL_MAP[tool_name](**tool_input)
                    result_content = json.dumps(result)
                    is_error = False
                except Exception as exc:
                    result_content = f"Error: {exc}"
                    is_error = True

                if verbose:
                    preview = result_content[:200]
                    print(f"  ← {preview}{'...' if len(result_content) > 200 else ''}")

                tool_results.append({
                    "type":       "tool_result",
                    "tool_use_id": block.id,
                    "content":    result_content,
                    "is_error":   is_error,
                })

            # Return tool results to Claude
            messages.append({"role": "user", "content": tool_results})
            continue

        # Unexpected stop reason
        raise RuntimeError(f"Unexpected stop_reason: {response.stop_reason}")


print("Agent loop ready.")


## 5. Demo 1 — Triage Unread Emails

Ask the agent to read the inbox and flag anything that needs an urgent reply.


In [ ]:
result = run_agent(
    "Read my unread emails and tell me which ones need an urgent reply. "
    "For each urgent message give me: sender, subject, one-sentence reason it's urgent, "
    "and a suggested reply deadline."
)

print("\n" + "=" * 60)
print(result)


## 6. Demo 2 — Draft and Send Pricing Follow-Ups

A multi-step task: search for pricing enquiries, read each one in full,
and send a personalised follow-up reply.


In [ ]:
result = run_agent(
    "Search for any emails where someone asked about pricing or plans. "
    "For each one, read the full email, then draft and send a polite follow-up that: "
    "(1) thanks them for their interest, (2) briefly lists our Starter ($29/mo), "
    "Pro ($79/mo), and Enterprise (custom) tiers, and (3) offers a 15-minute demo call. "
    "Use the sender's first name if you can infer it from the email."
)

print("\n" + "=" * 60)
print(result)


## 7. Advanced: Continuous Inbox Monitoring

For production deployments you'll want the agent to poll the inbox on a schedule
and act on new messages automatically.

The snippet below runs the triage loop every 5 minutes in a background thread.
In a real deployment, replace `threading` with a cron job, a task queue,
or an async event loop depending on your infrastructure.


In [ ]:
import threading
import time

POLL_INTERVAL_SECONDS = 300   # 5 minutes
_stop_event = threading.Event()


def monitoring_loop():
    """Poll the inbox every POLL_INTERVAL_SECONDS and triage new messages."""
    print("[monitor] Starting inbox monitoring loop.")
    while not _stop_event.is_set():
        print(f"[monitor] Checking inbox at {time.strftime('%H:%M:%S')} ...")
        try:
            summary = run_agent(
                "Check for any new unread emails received in the last 10 minutes. "
                "If there are none, just say 'No new messages.' "
                "Otherwise, give a one-line summary for each and flag any that look urgent.",
                verbose=False,
            )
            print(f"[monitor] {summary}")
        except Exception as exc:
            print(f"[monitor] Error: {exc}")

        _stop_event.wait(timeout=POLL_INTERVAL_SECONDS)

    print("[monitor] Monitoring loop stopped.")


# ── Start / stop helpers ────────────────────────────────────────────────────
def start_monitoring():
    _stop_event.clear()
    t = threading.Thread(target=monitoring_loop, daemon=True)
    t.start()
    return t


def stop_monitoring():
    _stop_event.set()


# Uncomment to start:
# monitor_thread = start_monitoring()
# # ... later ...
# stop_monitoring()

print("Monitoring helpers defined. Uncomment the last two lines to run.")


## 8. Conclusion

You've built a fully functional email agent that:

- Uses Claude's native **tool use** to decide when to read, search, or send mail
- Runs the complete **agentic loop** without any framework dependencies
- Integrates with **Commune** for real inbox access and delivery
- Can be extended to run continuously as a background monitoring service

### Next steps

- Add a `mark_read` tool so the agent can track what it has already processed
- Integrate with a CRM (HubSpot, Salesforce) by adding more tools
- Use Claude's [extended thinking](https://docs.anthropic.com/claude/docs/extended-thinking)
  for complex triage decisions
- Deploy as a serverless function triggered by Commune webhooks (no polling needed)

### Resources

- [Anthropic tool use docs](https://docs.anthropic.com/claude/docs/tool-use)
- [Commune API reference](https://commune.email/docs)
- [commune-mail PyPI package](https://pypi.org/project/commune-mail/)
- [Claude model overview](https://docs.anthropic.com/claude/docs/models-overview)
